# V4 GNN-TAT - Intra-Cell DEM Features

Retrains V4 GNN-TAT (Graph Neural Network with Temporal Attention) with
**sub-grid topographic heterogeneity features** from intra-cell DEM analysis.

**Baseline:** `models/base_models_gnn_tat_v4.ipynb` with BASIC/KCE/PAFC

**This notebook:** Same architecture, new feature bundles:
- `BASIC_D10`: BASIC + 10 elevation deciles (p10-p100) per cell
- `BASIC_PCA6`: BASIC + 6 PCA components of intra-cell DEM
- `BASIC_D10_STATS`: BASIC + 10 deciles + 5 statistics

**Architecture:** GAT spatial encoder + Multi-Head Temporal Attention + LSTM decoder
- Graph: 3,965 nodes (CHIRPS cells), edges based on distance + elevation + correlation
- Parameters: ~98K (34% fewer than ConvLSTM V2)
- Chunked GNN processing to avoid OOM on full 61x65 grid

**GPU:** A100 recommended. Training time ~4-6h per feature bundle.

## 1. Environment Setup

In [ ]:
# ==================================================
# ENVIRONMENT SETUP
# ==================================================

import os, sys, shutil

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running in Google Colab')
    from google.colab import drive
    drive.mount('/content/drive')

    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch-geometric', '-q'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'netcdf4', 'h5py', 'xarray', 'h5netcdf', '-q'], check=True)

    BASE_PATH = '/content/drive/MyDrive/ml_precipitation_prediction'

    # Copy dataset to local disk for faster I/O
    DRIVE_DATA = f'{BASE_PATH}/data/output/complete_dataset_extended_dem_features.nc'
    LOCAL_DATA = '/content/dataset_local.nc'
    if os.path.exists(DRIVE_DATA):
        if os.path.exists(LOCAL_DATA):
            os.remove(LOCAL_DATA)
        print('Copying dataset to local disk...')
        shutil.copy(DRIVE_DATA, LOCAL_DATA)
        USE_LOCAL_DATA = True
    else:
        print(f'WARNING: Dataset not found: {DRIVE_DATA}')
        USE_LOCAL_DATA = False
else:
    print('Running locally')
    BASE_PATH = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
    if not BASE_PATH:
        BASE_PATH = r'd:\github.com\ninja-marduk\ml_precipitation_prediction'
    USE_LOCAL_DATA = False
    LOCAL_DATA = None

print(f'Base path: {BASE_PATH}')
if BASE_PATH not in sys.path:
    sys.path.insert(0, BASE_PATH)

print('Setup complete!')

In [ ]:
# ==================================================
# IMPORTS
# ==================================================

os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torch_geometric.nn import GCNConv, GATConv, SAGEConv

from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr

import gc, json, time, logging
from datetime import datetime
from typing import Dict, List, Tuple
from collections import defaultdict

import matplotlib.pyplot as plt

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Configuration (Intra-Cell DEM)

In [ ]:
# ==================================================
# CONFIGURATION - INTRA-CELL DEM
# ==================================================

FEATURES_BASIC = [
    'year', 'month', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos',
    'max_daily_precipitation', 'min_daily_precipitation', 'daily_precipitation_std',
    'elevation', 'slope', 'aspect',
]

FEATURES_BASIC_D10 = FEATURES_BASIC + [
    'dem_p10', 'dem_p20', 'dem_p30', 'dem_p40', 'dem_p50',
    'dem_p60', 'dem_p70', 'dem_p80', 'dem_p90', 'dem_p100',
]

FEATURES_BASIC_PCA6 = FEATURES_BASIC + [
    'dem_pca_1', 'dem_pca_2', 'dem_pca_3',
    'dem_pca_4', 'dem_pca_5', 'dem_pca_6',
]

FEATURES_BASIC_D10_STATS = FEATURES_BASIC + [
    'dem_p10', 'dem_p20', 'dem_p30', 'dem_p40', 'dem_p50',
    'dem_p60', 'dem_p70', 'dem_p80', 'dem_p90', 'dem_p100',
    'dem_mean', 'dem_std', 'dem_skewness', 'dem_kurtosis', 'dem_range',
]

CONFIG = {
    'input_window': 60,
    'horizon': 12,
    'epochs': 150,
    'batch_size': 2,
    'learning_rate': 1e-3,
    'patience': 50,
    'light_mode': False,
    'light_grid_size': 5,
    'enabled_horizons': [12],
    'train_val_split': 0.8,
    'weight_decay': 1e-5,
    'base_path': Path(BASE_PATH),
    'feature_sets': {
        'BASIC_D10': FEATURES_BASIC_D10,       # 22 features
        'BASIC_PCA6': FEATURES_BASIC_PCA6,      # 18 features
        # 'BASIC_D10_STATS': FEATURES_BASIC_D10_STATS,  # uncomment if needed
    },
    'gnn_config': {
        'hidden_dim': 64,
        'num_gnn_layers': 2,
        'gnn_type': 'GAT',
        'num_heads': 4,
        'dropout': 0.1,
        'temporal_hidden_dim': 64,
        'num_temporal_heads': 4,
        'temporal_dropout': 0.1,
        'lstm_hidden_dim': 64,
        'num_lstm_layers': 2,
        'edge_threshold': 0.3,
        'max_neighbors': 8,
        'use_distance_edges': True,
        'use_elevation_edges': True,
        'use_correlation_edges': True,
        'distance_scale_km': 10.0,
        'elevation_scale': 0.2,
        'elevation_weight': 0.3,
        'correlation_weight': 0.5,
        'min_edge_weight': 0.01,
    }
}

# Dataset path (extended with DEM features)
if IN_COLAB and USE_LOCAL_DATA:
    CONFIG['data_file'] = Path(LOCAL_DATA)
else:
    CONFIG['data_file'] = CONFIG['base_path'] / 'data' / 'output' / (
        'complete_dataset_extended_dem_features.nc'
    )

CONFIG['out_root'] = CONFIG['base_path'] / 'models' / 'output' / 'V4_GNN_TAT_Models_intracell_dem'
CONFIG['out_root'].mkdir(parents=True, exist_ok=True)

print(f'Dataset: {CONFIG["data_file"]}')
print(f'Output: {CONFIG["out_root"]}')
print(f'GNN type: {CONFIG["gnn_config"]["gnn_type"]}')
for name, feats in CONFIG['feature_sets'].items():
    print(f'  {name}: {len(feats)} features')

## 3. Data Loading

In [ ]:
# ==================================================
# DATA LOADING
# ==================================================

def load_and_validate_data(config):
    data_file = config['data_file']
    if not Path(data_file).exists():
        raise FileNotFoundError(f'Dataset not found: {data_file}')

    engines = ['h5netcdf', 'netcdf4', 'scipy']
    ds = None
    for engine in engines:
        try:
            ds = xr.open_dataset(data_file, engine=engine)
            print(f'Loaded with engine: {engine}')
            break
        except Exception:
            continue
    if ds is None:
        raise RuntimeError('Could not open dataset')

    if config['light_mode']:
        g = config['light_grid_size']
        lc, lnc = len(ds.latitude) // 2, len(ds.longitude) // 2
        ds = ds.isel(latitude=slice(lc - g//2, lc - g//2 + g),
                     longitude=slice(lnc - g//2, lnc - g//2 + g))

    n_lat, n_lon = len(ds.latitude), len(ds.longitude)
    lat_coords, lon_coords = ds.latitude.values, ds.longitude.values

    # Validate features
    available = set(list(ds.data_vars) + list(ds.coords))
    for exp_name, feats in config['feature_sets'].items():
        missing = [f for f in feats if f not in available]
        if missing:
            raise ValueError(f'{exp_name} missing: {missing}')
        print(f'  {exp_name}: all {len(feats)} features present')

    print(f'Grid: {n_lat}x{n_lon} = {n_lat*n_lon} nodes, {len(ds.time)} timesteps')
    return ds, n_lat, n_lon, lat_coords, lon_coords

ds, lat, lon, lat_coords, lon_coords = load_and_validate_data(CONFIG)

## 4. Spatial Graph Construction

In [ ]:
# ==================================================
# SPATIAL GRAPH CONSTRUCTION
# ==================================================

class SpatialGraphBuilder:
    """Builds spatial graph from precipitation grid data."""

    def __init__(self, lat_coords, lon_coords, elevation, config):
        self.lat_coords = lat_coords
        self.lon_coords = lon_coords
        self.elevation = elevation
        self.config = config['gnn_config']
        self.n_lat = len(lat_coords)
        self.n_lon = len(lon_coords)
        self.n_nodes = self.n_lat * self.n_lon
        self.node_positions = np.array([[la, lo] for la in lat_coords for lo in lon_coords])
        self.node_elevations = elevation.flatten()
        print(f'SpatialGraphBuilder: {self.n_nodes} nodes ({self.n_lat}x{self.n_lon})')

    def compute_distance_matrix(self):
        pos_rad = np.radians(self.node_positions)
        lat1, lon1 = pos_rad[:, 0:1], pos_rad[:, 1:2]
        dlat = lat1.T - lat1
        dlon = lon1.T - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat1.T) * np.sin(dlon/2)**2
        return 6371.0 * 2 * np.arcsin(np.sqrt(a))

    def compute_elevation_similarity(self):
        elev = self.node_elevations.reshape(-1, 1)
        elev_diff = np.abs(elev - elev.T)
        elev_range = elev.max() - elev.min() + 1e-6
        scale = self.config.get('elevation_scale', 0.2)
        return np.exp(-elev_diff / (elev_range * scale))

    def compute_correlation_matrix(self, precip_series):
        T = precip_series.shape[0]
        flat = precip_series.reshape(T, -1)
        flat = np.nan_to_num(flat, nan=0.0)
        centered = flat - flat.mean(axis=0, keepdims=True)
        std = flat.std(axis=0, keepdims=True) + 1e-8
        norm = centered / std
        return np.clip((norm.T @ norm) / T, -1, 1)

    def build_adjacency_matrix(self, precip_series=None):
        cfg = self.config
        adj = np.zeros((self.n_nodes, self.n_nodes))

        if cfg['use_distance_edges']:
            dist = self.compute_distance_matrix()
            dist[dist == 0] = 1e-6
            sim = 1.0 / (1.0 + dist / cfg.get('distance_scale_km', 10.0))
            for i in range(self.n_nodes):
                neighbors = np.argsort(dist[i])[:cfg['max_neighbors'] + 1]
                neighbors = neighbors[neighbors != i][:cfg['max_neighbors']]
                adj[i, neighbors] += sim[i, neighbors]
            print(f'  Distance edges added (k={cfg["max_neighbors"]})')

        if cfg['use_elevation_edges']:
            adj += self.compute_elevation_similarity() * cfg.get('elevation_weight', 0.3)
            print('  Elevation edges added')

        if cfg['use_correlation_edges'] and precip_series is not None:
            corr = self.compute_correlation_matrix(precip_series)
            adj += np.maximum(corr - cfg['edge_threshold'], 0) * cfg.get('correlation_weight', 0.5)
            print('  Correlation edges added')

        np.fill_diagonal(adj, 0)
        adj_max = adj.max()
        if adj_max > 0:
            adj = adj / adj_max
        adj = (adj + adj.T) / 2

        # Vectorized edge extraction (replaces slow double for-loop)
        min_w = cfg.get('min_edge_weight', 0.01)
        rows, cols = np.where(adj > min_w)
        edge_index = np.stack([rows, cols], axis=0)
        edge_weight = adj[rows, cols]
        print(f'  Edges: {len(edge_weight)}, avg degree: {len(edge_weight)/self.n_nodes:.1f}')

        # Limit edges if too many
        max_edges = 500_000
        if len(edge_weight) > max_edges:
            top_idx = np.argsort(edge_weight)[-max_edges:]
            edge_index = edge_index[:, top_idx]
            edge_weight = edge_weight[top_idx]
            print(f'  Reduced to {len(edge_weight)} edges')

        return edge_index, edge_weight


# Build graph
print('Building spatial graph...')
elevation = ds['elevation'].values
if elevation.ndim == 3:
    elevation = elevation[0]
precip_series = ds['total_precipitation'].values

graph_builder = SpatialGraphBuilder(lat_coords, lon_coords, elevation, CONFIG)
edge_index, edge_weight = graph_builder.build_adjacency_matrix(precip_series)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
edge_index_tensor = torch.tensor(edge_index, dtype=torch.long).to(device)
edge_weight_tensor = torch.tensor(edge_weight, dtype=torch.float32).to(device)
print(f'Graph on {device}: edge_index={edge_index_tensor.shape}, edge_weight={edge_weight_tensor.shape}')

## 5. Data Preprocessing

In [ ]:
# ==================================================
# DATA PREPROCESSING
# ==================================================

def windowed_arrays(X, y, input_window, horizon, start_indices=None):
    seq_X, seq_y = [], []
    T = len(X)
    iterator = start_indices if start_indices is not None else range(T - input_window - horizon + 1)
    for start in iterator:
        if start < 0:
            continue
        end_w = start + input_window
        end_y = end_w + horizon
        if end_y > T:
            continue
        Xw, yw = X[start:end_w], y[end_w:end_y]
        if np.isnan(Xw).any() or np.isnan(yw).any():
            continue
        seq_X.append(Xw)
        seq_y.append(yw)
    if not seq_X:
        raise ValueError('No valid windows')
    return np.asarray(seq_X, np.float32), np.asarray(seq_y, np.float32)


def compute_split_indices(total_steps, input_window, horizon, split_ratio):
    cutoff = max(0, int(total_steps * split_ratio))
    max_start = total_steps - input_window - horizon + 1
    train_end = max(0, cutoff - input_window - horizon + 1)
    train_indices = list(range(0, train_end))
    val_start = min(max_start, max(cutoff, 0))
    val_indices = list(range(val_start, max_start))
    return train_indices, val_indices


def fill_nan_with_median(X, feature_list):
    """Fill NaN in features with per-feature spatial median.

    603 cells (15.2%) have NaN in DEM features because the 90m DEM GeoTIFF
    doesn't cover those cells. Filling with median makes these cells 'average'
    after StandardScaler, minimizing bias (~200mm bias avoided vs fill=0).
    """
    nan_count = 0
    for i, feat in enumerate(feature_list):
        if np.isnan(X[..., i]).any():
            valid_vals = X[0, :, :, i][~np.isnan(X[0, :, :, i])]
            fill_val = float(np.median(valid_vals)) if len(valid_vals) > 0 else 0.0
            X[..., i] = np.nan_to_num(X[..., i], nan=fill_val)
            nan_count += 1
            print(f'    Filled {feat} NaN with median={fill_val:.1f}')
    if nan_count > 0:
        print(f'    Filled NaN in {nan_count} features')
    return X


def preprocess_data(ds, config, n_lat, n_lon, horizon):
    results = {}
    for exp_name, feature_list in config['feature_sets'].items():
        print(f'\nProcessing {exp_name}...')
        arrays = []
        for feat in feature_list:
            # Check both data_vars and coords (year, month may be coords)
            if feat in ds.data_vars:
                arr = ds[feat].values
            elif feat in ds.coords:
                arr = ds[feat].values
            else:
                print(f'  WARNING: feature {feat} not found, skipping')
                continue
            if arr.ndim == 2:
                arr = np.broadcast_to(arr, (len(ds.time), n_lat, n_lon))
            elif arr.ndim == 1:
                arr = np.broadcast_to(arr[:, np.newaxis, np.newaxis], (len(ds.time), n_lat, n_lon))
            arrays.append(arr)

        X = np.stack(arrays, axis=-1).astype(np.float32)
        y = ds['total_precipitation'].values.astype(np.float32)

        # Fill NaN in DEM features with per-feature spatial median (not 0)
        X = fill_nan_with_median(X, feature_list)
        y = np.nan_to_num(y, nan=0.0)

        train_idx, val_idx = compute_split_indices(
            len(ds.time), config['input_window'], horizon, config['train_val_split']
        )

        X_tr, y_tr = windowed_arrays(X, y, config['input_window'], horizon, train_idx)
        X_va, y_va = windowed_arrays(X, y, config['input_window'], horizon, val_idx)

        # Scale features
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        scaler_X.fit(X_tr.reshape(-1, X_tr.shape[-1]))
        scaler_y.fit(y_tr.reshape(-1, 1))

        X_tr_sc = scaler_X.transform(X_tr.reshape(-1, X_tr.shape[-1])).reshape(X_tr.shape)
        X_va_sc = scaler_X.transform(X_va.reshape(-1, X_va.shape[-1])).reshape(X_va.shape)
        y_tr_sc = scaler_y.transform(y_tr.reshape(-1, 1)).reshape(y_tr.shape)
        y_va_sc = scaler_y.transform(y_va.reshape(-1, 1)).reshape(y_va.shape)

        y_tr_sc = np.expand_dims(y_tr_sc, -1)
        y_va_sc = np.expand_dims(y_va_sc, -1)

        results[exp_name] = (X_tr_sc, y_tr_sc, X_va_sc, y_va_sc, scaler_y)
        print(f'  {exp_name}: X_tr={X_tr_sc.shape}, y_tr={y_tr_sc.shape}')

    return results

data_splits = preprocess_data(ds, CONFIG, lat, lon, CONFIG['horizon'])
print(f'\nReady: {list(data_splits.keys())}')

## 6. GNN-TAT Model Architecture

In [ ]:
# ==================================================
# GNN-TAT MODEL ARCHITECTURE
# ==================================================

class TemporalAttention(nn.Module):
    """Multi-Head Temporal Attention with residual connection."""
    def __init__(self, input_dim, hidden_dim, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.hidden_dim = hidden_dim
        assert hidden_dim % num_heads == 0

        self.q_proj = nn.Linear(input_dim, hidden_dim)
        self.k_proj = nn.Linear(input_dim, hidden_dim)
        self.v_proj = nn.Linear(input_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.residual_proj = nn.Linear(input_dim, hidden_dim) if input_dim != hidden_dim else nn.Identity()
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5

    def forward(self, x):
        B, S, _ = x.shape
        residual = self.residual_proj(x)
        Q = self.q_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * self.scale, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, S, self.hidden_dim)
        return self.layer_norm(residual + self.dropout(self.out_proj(out)))


class SpatialGNNEncoder(nn.Module):
    """GNN Encoder supporting GAT, SAGE, GCN."""
    def __init__(self, input_dim, hidden_dim, num_layers, gnn_type='GAT', num_heads=4, dropout=0.1):
        super().__init__()
        self.gnn_type = gnn_type
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.gnn_layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            if gnn_type == 'GAT':
                layer = GATConv(hidden_dim, hidden_dim // num_heads, heads=num_heads, dropout=dropout, concat=True)
            elif gnn_type == 'SAGE':
                layer = SAGEConv(hidden_dim, hidden_dim, aggr='mean')
            else:
                layer = GCNConv(hidden_dim, hidden_dim, add_self_loops=True, normalize=True)
            self.gnn_layers.append(layer)
            self.norms.append(nn.LayerNorm(hidden_dim))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_weight=None):
        x = self.input_proj(x)
        for gnn, norm in zip(self.gnn_layers, self.norms):
            residual = x
            if self.gnn_type == 'GCN':
                x = gnn(x, edge_index, edge_weight)
            else:
                x = gnn(x, edge_index)
            # Match original: gelu -> dropout -> norm(x + residual)
            x = F.gelu(x)
            x = self.dropout(x)
            x = norm(x + residual)
        return x


class GNN_TAT(nn.Module):
    """Graph Neural Network with Temporal Attention (memory-optimized)."""
    def __init__(self, n_features, n_nodes, n_lat, n_lon, horizon, config, gnn_chunk_size=15):
        super().__init__()
        self.n_features = n_features
        self.n_nodes = n_nodes
        self.n_lat = n_lat
        self.n_lon = n_lon
        self.horizon = horizon
        self.gnn_chunk_size = gnn_chunk_size

        cfg = config['gnn_config']
        hidden_dim = cfg['hidden_dim']
        self.hidden_dim = hidden_dim

        self.gnn_encoder = SpatialGNNEncoder(
            n_features, hidden_dim, cfg['num_gnn_layers'],
            cfg['gnn_type'], cfg['num_heads'], cfg['dropout']
        )
        self.temporal_attention = TemporalAttention(
            hidden_dim, hidden_dim, cfg['num_temporal_heads'], cfg['temporal_dropout']
        )
        self.lstm = nn.LSTM(
            hidden_dim, hidden_dim, cfg['num_lstm_layers'],
            batch_first=True, dropout=cfg['dropout'] if cfg['num_lstm_layers'] > 1 else 0
        )
        self.output_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Dropout(cfg['dropout']), nn.Linear(hidden_dim, horizon)
        )

        total_params = sum(p.numel() for p in self.parameters())
        print(f'GNN-TAT: {n_features} features, {n_nodes} nodes, {total_params:,} params')
        print(f'  Chunk size: {gnn_chunk_size} timesteps')

    def _batch_edge_index(self, edge_index, num_graphs, device):
        num_edges = edge_index.shape[1]
        offsets = torch.arange(num_graphs, device=device).view(-1, 1, 1) * self.n_nodes
        batch_ei = edge_index.unsqueeze(0).expand(num_graphs, -1, -1) + offsets.expand(-1, 2, num_edges)
        return batch_ei.permute(1, 0, 2).reshape(2, -1)

    def _process_gnn_chunk(self, x_chunk, edge_index, edge_weight=None):
        chunk_size = x_chunk.shape[0]
        batch_ei = self._batch_edge_index(edge_index, chunk_size, x_chunk.device)
        batch_ew = edge_weight.repeat(chunk_size) if edge_weight is not None else None
        x_nodes = x_chunk.view(-1, self.n_features)
        return self.gnn_encoder(x_nodes, batch_ei, batch_ew).view(chunk_size, self.n_nodes, self.hidden_dim)

    def forward(self, x, edge_index, edge_weight=None):
        B, S = x.shape[:2]
        x = x.view(B, S, self.n_nodes, self.n_features)
        x_flat = x.view(B * S, self.n_nodes, self.n_features)
        total = B * S

        # Chunked GNN
        outputs = []
        for i in range(0, total, self.gnn_chunk_size):
            outputs.append(self._process_gnn_chunk(x_flat[i:min(i+self.gnn_chunk_size, total)], edge_index, edge_weight))
        gnn_out = torch.cat(outputs, dim=0).view(B, S, self.n_nodes, self.hidden_dim)
        del outputs

        # Temporal processing per node
        temporal_in = gnn_out.permute(0, 2, 1, 3).reshape(B * self.n_nodes, S, self.hidden_dim)
        temporal_out = self.temporal_attention(temporal_in)
        lstm_out, _ = self.lstm(temporal_out)
        out = self.output_proj(lstm_out[:, -1, :])

        # Reshape to grid
        out = out.view(B, self.n_nodes, self.horizon).permute(0, 2, 1)
        return out.view(B, self.horizon, self.n_lat, self.n_lon, 1)


# Determine chunk size
n_nodes = lat * lon
if n_nodes <= 50:
    GNN_CHUNK_SIZE = 60
elif n_nodes <= 500:
    GNN_CHUNK_SIZE = 8
else:
    GNN_CHUNK_SIZE = 2

print(f'Grid: {n_nodes} nodes, chunk_size={GNN_CHUNK_SIZE}')

## 7. Training Functions

In [ ]:
# ==================================================
# TRAINING FUNCTIONS
# ==================================================

class PrecipitationDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def train_pytorch_model(model, X_tr, y_tr, X_va, y_va,
                        edge_index, edge_weight, config,
                        model_name, exp_name, out_dir, horizon, device):
    """Train PyTorch GNN-TAT model."""
    metrics_dir = out_dir / f'h{horizon}' / exp_name / 'training_metrics'
    metrics_dir.mkdir(parents=True, exist_ok=True)
    model_path = metrics_dir / f'{model_name}_best_h{horizon}.pt'

    train_loader = DataLoader(PrecipitationDataset(X_tr, y_tr), batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(PrecipitationDataset(X_va, y_va), batch_size=config['batch_size'], shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'],
                                  weight_decay=config.get('weight_decay', 1e-5))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=config['patience']//2)
    criterion = nn.MSELoss()

    history = {'train_loss': [], 'val_loss': [], 'lr': []}
    best_val_loss = float('inf')
    patience_counter = 0
    best_epoch = 0

    print(f'\nTraining {model_name} on {exp_name}...')

    for epoch in range(config['epochs']):
        model.train()
        train_losses = []
        for bX, by in train_loader:
            bX, by = bX.to(device), by.to(device)
            optimizer.zero_grad()
            output = model(bX, edge_index, edge_weight)
            loss = criterion(output, by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for bX, by in val_loader:
                bX, by = bX.to(device), by.to(device)
                val_losses.append(criterion(model(bX, edge_index, edge_weight), by).item())

        epoch_train = np.mean(train_losses)
        epoch_val = np.mean(val_losses)
        lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(epoch_train)
        history['val_loss'].append(epoch_val)
        history['lr'].append(lr)
        scheduler.step(epoch_val)

        if epoch_val < best_val_loss:
            best_val_loss = epoch_val
            best_epoch = epoch
            patience_counter = 0
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'val_loss': best_val_loss}, model_path)
        else:
            patience_counter += 1

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1:3d}/{config["epochs"]}: train={epoch_train:.4f}, '
                  f'val={epoch_val:.4f}, lr={lr:.2e}')

        if patience_counter >= config['patience']:
            print(f'  Early stopping at epoch {epoch+1}')
            break

    # Load best
    checkpoint = torch.load(model_path, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])

    # Save history
    pd.DataFrame(history).to_csv(metrics_dir / f'{model_name}_training_log_h{horizon}.csv', index=False)
    summary = {
        'model_name': model_name, 'experiment': exp_name, 'horizon': horizon,
        'best_epoch': best_epoch + 1, 'best_val_loss': float(best_val_loss),
        'total_epochs': len(history['train_loss']),
        'parameters': sum(p.numel() for p in model.parameters()),
    }
    with open(metrics_dir / f'{model_name}_history.json', 'w') as f:
        json.dump(summary, f, indent=2)

    print(f'  Best val_loss: {best_val_loss:.4f} at epoch {best_epoch+1}')
    return model, summary


def evaluate_model(model, X_va, y_va, edge_index, edge_weight, scaler, device, horizon):
    """Evaluate and return metrics + inverse-transformed predictions."""
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_va), 4):
            bX = torch.tensor(X_va[i:i+4], dtype=torch.float32).to(device)
            preds.append(model(bX, edge_index, edge_weight).cpu().numpy())
    y_hat_sc = np.concatenate(preds, axis=0)
    y_hat = scaler.inverse_transform(y_hat_sc.reshape(-1, 1)).reshape(y_hat_sc.shape)
    y_true = scaler.inverse_transform(y_va.reshape(-1, 1)).reshape(y_va.shape)

    from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
    results = []
    for h in range(horizon):
        t = y_true[:, h].flatten()
        p = y_hat[:, h].flatten()
        results.append({
            'H': h + 1,
            'RMSE': np.sqrt(mean_squared_error(t, p)),
            'MAE': mean_absolute_error(t, p),
            'R2': r2_score(t, p),
        })
    return results, y_hat, y_true


print('Training functions defined')

## 8. Main Training Loop

In [ ]:
# ==================================================
# MAIN TRAINING LOOP - Only GAT (primary GNN-TAT)
# ==================================================

all_results = []
all_predictions = {}

for horizon in CONFIG['enabled_horizons']:
    print(f'\n{"="*60}')
    print(f'Training for H={horizon}')
    print(f'{"="*60}')

    for exp_name in CONFIG['feature_sets']:
        if exp_name not in data_splits:
            continue

        X_tr, y_tr, X_va, y_va, scaler = data_splits[exp_name]
        n_features = X_tr.shape[-1]
        model_name = 'GNN_TAT_GAT'

        print(f'\n--- {exp_name}: {n_features} features, model={model_name} ---')

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        try:
            model = GNN_TAT(
                n_features=n_features,
                n_nodes=graph_builder.n_nodes,
                n_lat=lat, n_lon=lon,
                horizon=horizon,
                config=CONFIG,
                gnn_chunk_size=GNN_CHUNK_SIZE
            ).to(device)

            model, summary = train_pytorch_model(
                model, X_tr, y_tr, X_va, y_va,
                edge_index_tensor, edge_weight_tensor,
                CONFIG, model_name, exp_name,
                CONFIG['out_root'], horizon, device
            )

            # Evaluate
            results, y_hat, y_true = evaluate_model(
                model, X_va, y_va,
                edge_index_tensor, edge_weight_tensor,
                scaler, device, horizon
            )

            for r in results:
                r['Experiment'] = exp_name
                r['Model'] = model_name
            all_results.extend(results)

            avg_r2 = np.mean([r['R2'] for r in results])
            avg_rmse = np.mean([r['RMSE'] for r in results])
            print(f'  Overall: R2={avg_r2:.4f}, RMSE={avg_rmse:.2f}mm')

            # Export predictions for V10 Late Fusion
            export_dir = CONFIG['out_root'] / 'map_exports' / f'H{horizon}' / exp_name / model_name
            export_dir.mkdir(parents=True, exist_ok=True)
            np.save(export_dir / 'predictions.npy', y_hat.astype(np.float32))
            # Targets from V2 notebook (same test set)
            np.save(export_dir / 'targets.npy', y_true.astype(np.float32))
            metadata = {
                'model': model_name, 'experiment': exp_name, 'horizon': horizon,
                'shape': list(y_hat.shape),
                'r2_mean': float(avg_r2), 'rmse_mean': float(avg_rmse),
                'parameters': sum(p.numel() for p in model.parameters()),
                'generated_at': datetime.now().isoformat(),
            }
            with open(export_dir / 'metadata.json', 'w') as f:
                json.dump(metadata, f, indent=2)
            print(f'  Saved: {export_dir}')

        except Exception as e:
            print(f'ERROR: {e}')
            import traceback
            traceback.print_exc()
        finally:
            if 'model' in dir():
                del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

# Save results
if all_results:
    res_df = pd.DataFrame(all_results)
    out_csv = CONFIG['out_root'] / 'metrics_spatial_v4_intracell_dem_h12.csv'
    res_df.to_csv(out_csv, index=False)
    print(f'\nResults saved: {out_csv}')
    print(res_df.groupby(['Experiment', 'Model'])[['RMSE', 'MAE', 'R2']].mean().round(4))

print('\nTraining complete!')

## 9. Summary and Verification

In [ ]:
# ==================================================
# SUMMARY AND VERIFICATION
# ==================================================

from sklearn.metrics import r2_score

print('=' * 60)
print('V4 GNN-TAT INTRACELL DEM TRAINING COMPLETE')
print('=' * 60)

for horizon in CONFIG['enabled_horizons']:
    for exp_name in CONFIG['feature_sets']:
        pred_dir = CONFIG['out_root'] / 'map_exports' / f'H{horizon}' / exp_name / 'GNN_TAT_GAT'
        pred_file = pred_dir / 'predictions.npy'
        targ_file = pred_dir / 'targets.npy'

        if pred_file.exists() and targ_file.exists():
            pred = np.load(pred_file)
            targ = np.load(targ_file)
            r2 = r2_score(targ.ravel(), pred.ravel())
            rmse = np.sqrt(np.mean((pred - targ)**2))
            print(f'\n{exp_name} (H{horizon}):')
            print(f'  predictions: {pred.shape}')
            print(f'  targets:     {targ.shape}')
            print(f'  R2={r2:.4f}, RMSE={rmse:.2f}mm')
        else:
            print(f'\n{exp_name} (H{horizon}): MISSING predictions!')

print('\n' + '=' * 60)
print('Next steps:')
print('1. Run V10 fusion: evaluate_v10_fusion_intracell_dem.ipynb')
print('2. Or locally: python workflows/run_pipeline.py --from 7 --intracell-dem')
print('=' * 60)